# Does Weather Affect the Stock Market? A BIST 100 Analysis
**DSA210 – Data Collection & EDA**

## 1. BIST 100 Data

In [ ]:
import yfinance as yf
import pandas as pd
import os

os.makedirs("../data", exist_ok=True)

bist = yf.download("XU100.IS", start="2023-01-01", end="2024-12-31")
bist = bist[["Open", "Close", "Volume"]].copy()
bist.columns = ["open", "close", "volume"]
bist["daily_return"] = (bist["close"] - bist["open"]) / bist["open"] * 100
bist.index = pd.to_datetime(bist.index)
bist.index.name = "date"

bist.to_csv("../data/bist100.csv")
print("Saved:", len(bist), "trading days")
bist.head()

## 2. Istanbul Weather Data

In [ ]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 41.0082,
    "longitude": 28.9784,
    "start_date": "2023-01-01",
    "end_date": "2024-12-31",
    "daily": "temperature_2m_mean,precipitation_sum,windspeed_10m_max",
    "format": "csv"
}

response = requests.get(url, params=params)

with open("../data/istanbul_weather.csv", "w", encoding="utf-8") as f:
    f.write(response.text)

weather = pd.read_csv("../data/istanbul_weather.csv", skiprows=3)
weather.columns = ["date", "temp_mean", "precipitation", "windspeed"]
weather["date"] = pd.to_datetime(weather["date"])
weather = weather.set_index("date")

print("Saved:", len(weather), "days")
weather.head()

## 3. Merge Datasets

In [ ]:
df = bist.join(weather, how="inner")
df.to_csv("../data/merged.csv")
print("Merged dataset:", df.shape)
df.head()

## 4. Machine Learning

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, classification_report

X = df[["temp_mean", "precipitation", "windspeed"]]
y = df["daily_return"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# linear regression
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("R2:", round(model.score(X_test, y_test), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, preds)), 4))

In [ ]:
# logistic regression - predict up or down
y_dir = (df["daily_return"] > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y_dir, test_size=0.2, random_state=42)

clf = LogisticRegression()
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test)))